# 🏀 From Combine to Career: What Pre-Draft Measurements Really Predict

> **Does wingspan predict All-Star selection? Does lane agility predict career longevity?**
> The first public analysis joining combine metrics with full career data.

**Part 4 of 10** in the [wyattowalsh/basketball](https://www.kaggle.com/datasets/wyattowalsh/basketball) companion notebook series.

---

## What makes this analysis unique?

No other public NBA dataset joins pre-draft combine measurements (wingspan, vertical leap, lane agility, sprint times) with complete career statistics. This notebook exploits that join to answer:

1. How has the combine athlete evolved over decades?
2. How steeply does draft pick value drop off?
3. Which physical measurements actually correlate with career success?
4. Can we predict career longevity from combine data? (Survival analysis)
5. Can we predict stars vs busts from combine data? (CatBoost classifier)
6. Who most outperformed or underperformed their combine profile?

---

## Table of Contents

1. [Setup & Data Loading](#1-setup--data-loading)
2. [Combine Data Exploration](#2-combine-data-exploration)
3. [Draft Pick Value Curve](#3-draft-pick-value-curve)
4. [Correlation Analysis](#4-correlation-analysis)
5. [Survival Analysis: Career Longevity](#5-survival-analysis-career-longevity)
6. [Star Prediction: CatBoost Classifier](#6-star-prediction-catboost-classifier)
7. [Biggest Gaps: Outperformers & Underperformers](#7-biggest-gaps-outperformers--underperformers)
8. [Conclusion & Cross-Links](#8-conclusion--cross-links)

In [ ]:
!pip install -q "duckdb>=1.4,<2" "polars>=1.38,<2" "plotly>=5.18,<6" "scikit-learn>=1.4,<2" "shap>=0.44,<1" "catboost>=1.2,<2" "lifelines>=0.29,<1" "matplotlib>=3.8,<4"

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings("ignore")

import duckdb
import numpy as np
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from catboost import CatBoostClassifier, Pool
import shap

from lifelines import KaplanMeierFitter, CoxPHFitter
import matplotlib.pyplot as plt
from pathlib import Path

# Import shared utilities
_kaggle_path = Path("/kaggle/input/basketball")
sys.path.insert(0, str(_kaggle_path if _kaggle_path.exists() else Path(".")))
from nbadb_utils import COLORS, TEMPLATE, takeaway, get_connection

conn = get_connection()
print("Setup complete.")

---
## 2. Combine Data Exploration

The NBA Draft Combine has measured prospects since the mid-2000s. Let's see what data
we have and how the physical profile of the combine athlete has changed over time.

In [ ]:
# Load draft data with combine measurements
draft_df = conn.sql("""
    SELECT
        d.person_id,
        d.player_name,
        d.season,
        d.round_number,
        d.round_pick,
        d.overall_pick,
        d.team_id,
        d.organization,
        d.height_wo_shoes,
        d.height_w_shoes,
        d.weight,
        d.wingspan,
        d.standing_reach,
        d.body_fat_pct,
        d.hand_length,
        d.hand_width,
        d.standing_vertical_leap,
        d.max_vertical_leap,
        d.lane_agility_time,
        d.three_quarter_sprint,
        d.bench_press
    FROM fact_draft d
    ORDER BY d.season, d.overall_pick
""").pl()

# Filter to players with combine data (at least wingspan measured)
combine_df = draft_df.filter(pl.col("wingspan").is_not_null())

print(f"Total draft records: {len(draft_df):,}")
print(f"Records with combine data: {len(combine_df):,}")
print(f"Seasons covered: {combine_df['season'].min()} - {combine_df['season'].max()}")
print()

# Missingness per measurement column
measure_cols = [
    "height_wo_shoes", "weight", "wingspan", "standing_reach",
    "body_fat_pct", "hand_length", "hand_width",
    "standing_vertical_leap", "max_vertical_leap",
    "lane_agility_time", "three_quarter_sprint", "bench_press"
]

print("Missingness (among players with wingspan data):")
for col in measure_cols:
    null_pct = combine_df[col].null_count() / len(combine_df) * 100
    print(f"  {col:30s}: {null_pct:5.1f}% missing")

In [ ]:
# Derive decade and ratio columns
combine_df = combine_df.with_columns([
    (pl.col("season") // 10 * 10).cast(pl.Utf8).alias("decade"),
    (pl.col("wingspan") / pl.col("height_wo_shoes")).alias("wingspan_to_height"),
    (pl.col("standing_reach") / pl.col("height_wo_shoes")).alias("reach_to_height"),
])

# Box plots for key combine metrics by decade
box_metrics = [
    ("height_wo_shoes", "Height Without Shoes (in)"),
    ("weight", "Weight (lbs)"),
    ("wingspan", "Wingspan (in)"),
    ("body_fat_pct", "Body Fat %"),
    ("standing_vertical_leap", "Standing Vertical (in)"),
    ("max_vertical_leap", "Max Vertical (in)"),
    ("lane_agility_time", "Lane Agility (sec)"),
    ("three_quarter_sprint", "3/4 Court Sprint (sec)"),
    ("bench_press", "Bench Press (reps)"),
    ("wingspan_to_height", "Wingspan / Height Ratio"),
]

fig = make_subplots(
    rows=5, cols=2,
    subplot_titles=[title for _, title in box_metrics],
    vertical_spacing=0.06,
    horizontal_spacing=0.08,
)

for idx, (col, title) in enumerate(box_metrics):
    row = idx // 2 + 1
    col_idx = idx % 2 + 1
    subset = combine_df.filter(pl.col(col).is_not_null())
    for i, decade in enumerate(sorted(subset["decade"].unique().to_list())):
        dec_data = subset.filter(pl.col("decade") == decade)
        fig.add_trace(
            go.Box(
                y=dec_data[col].to_list(),
                name=f"{decade}s",
                marker_color=[COLORS["primary"], COLORS["secondary"],
                              COLORS["accent"], COLORS["success"]][i % 4],
                showlegend=(idx == 0),
                legendgroup=f"{decade}s",
            ),
            row=row, col=col_idx,
        )

fig.update_layout(
    height=1600, width=1000,
    title_text="NBA Draft Combine Measurements by Decade",
    showlegend=True,
)
fig.show()

In [ ]:
# Summary statistics for derived ratios
ratio_summary = combine_df.select([
    pl.col("wingspan_to_height").mean().alias("mean_wingspan_ratio"),
    pl.col("wingspan_to_height").std().alias("std_wingspan_ratio"),
    pl.col("reach_to_height").mean().alias("mean_reach_ratio"),
    pl.col("reach_to_height").std().alias("std_reach_ratio"),
])
print("Derived ratio summary:")
print(ratio_summary)

takeaway(
    "The average NBA combine participant has a wingspan-to-height ratio of ~1.06, "
    "meaning their wingspan exceeds their height by about 6%. Athletes have gotten "
    "heavier over decades while vertical leap measurements have remained remarkably stable, "
    "suggesting the league selects for explosiveness regardless of era."
)

---
## 3. Draft Pick Value Curve

How much does draft position matter? We join draft data with career aggregates to
see how expected career value drops with each pick.

In [ ]:
# Join draft with career stats
draft_career = conn.sql("""
    SELECT
        d.person_id,
        d.player_name,
        d.season AS draft_season,
        d.round_number,
        d.overall_pick,
        c.career_gp,
        c.career_pts,
        c.career_ppg,
        c.career_rpg,
        c.career_apg,
        c.seasons_played
    FROM fact_draft d
    INNER JOIN agg_player_career c ON d.person_id = c.player_id
    WHERE d.overall_pick IS NOT NULL
      AND d.overall_pick > 0
    ORDER BY d.overall_pick
""").pl()

print(f"Draft picks with career data: {len(draft_career):,}")

# Mean career PPG by overall pick
pick_value = (
    draft_career
    .group_by("overall_pick")
    .agg([
        pl.col("career_ppg").mean().alias("mean_ppg"),
        pl.col("career_gp").mean().alias("mean_gp"),
        pl.col("career_pts").mean().alias("mean_pts"),
        pl.col("seasons_played").mean().alias("mean_seasons"),
        pl.col("career_ppg").count().alias("n_players"),
    ])
    .filter(pl.col("overall_pick") <= 60)
    .sort("overall_pick")
)

# Rolling average for trend line
pick_value = pick_value.with_columns([
    pl.col("mean_ppg").rolling_mean(window_size=5, center=True).alias("ppg_trend"),
    pl.col("mean_gp").rolling_mean(window_size=5, center=True).alias("gp_trend"),
])

In [ ]:
# Plot 1: PPG by draft pick
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=pick_value["overall_pick"].to_list(),
    y=pick_value["mean_ppg"].to_list(),
    mode="markers",
    name="Mean Career PPG",
    marker=dict(color=COLORS["primary"], size=7, opacity=0.6),
    hovertemplate="Pick %{x}<br>Mean PPG: %{y:.1f}<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=pick_value["overall_pick"].to_list(),
    y=pick_value["ppg_trend"].to_list(),
    mode="lines",
    name="5-Pick Rolling Avg",
    line=dict(color=COLORS["secondary"], width=3),
))
fig.add_vline(x=14, line_dash="dash", line_color=COLORS["muted"],
              annotation_text="Pick 14")
fig.update_layout(
    title="Draft Pick Value Curve: Career PPG by Overall Pick",
    xaxis_title="Overall Pick",
    yaxis_title="Mean Career PPG",
    height=500,
)
fig.show()

# Plot 2: Games played by draft pick
fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=pick_value["overall_pick"].to_list(),
    y=pick_value["mean_gp"].to_list(),
    mode="markers",
    name="Mean Career GP",
    marker=dict(color=COLORS["success"], size=7, opacity=0.6),
    hovertemplate="Pick %{x}<br>Mean GP: %{y:.0f}<extra></extra>",
))
fig2.add_trace(go.Scatter(
    x=pick_value["overall_pick"].to_list(),
    y=pick_value["gp_trend"].to_list(),
    mode="lines",
    name="5-Pick Rolling Avg",
    line=dict(color=COLORS["secondary"], width=3),
))
fig2.add_vline(x=14, line_dash="dash", line_color=COLORS["muted"],
               annotation_text="Pick 14")
fig2.update_layout(
    title="Draft Pick Value Curve: Career Games Played by Overall Pick",
    xaxis_title="Overall Pick",
    yaxis_title="Mean Career Games Played",
    height=500,
)
fig2.show()

takeaway(
    "There is a steep drop-off in expected career value after approximately pick 14. "
    "The top ~5 picks average nearly double the career PPG of picks 15-30, and the "
    "gap in career games played is even more dramatic. After the lottery, expected "
    "career production flattens considerably."
)

---
## 4. Correlation Analysis

Which combine measurements actually predict career success? We compute Pearson
correlations between every measurement and key career outcomes.

In [ ]:
# Build combined dataset: combine metrics + career outcomes
combined = conn.sql("""
    SELECT
        d.person_id,
        d.player_name,
        d.overall_pick,
        d.height_wo_shoes,
        d.weight,
        d.wingspan,
        d.standing_reach,
        d.body_fat_pct,
        d.hand_length,
        d.hand_width,
        d.standing_vertical_leap,
        d.max_vertical_leap,
        d.lane_agility_time,
        d.three_quarter_sprint,
        d.bench_press,
        c.career_gp,
        c.career_pts,
        c.career_ppg,
        c.seasons_played
    FROM fact_draft d
    INNER JOIN agg_player_career c ON d.person_id = c.player_id
    WHERE d.wingspan IS NOT NULL
""").pl()

# Add derived ratios
combined = combined.with_columns([
    (pl.col("wingspan") / pl.col("height_wo_shoes")).alias("wingspan_to_height"),
    (pl.col("standing_reach") / pl.col("height_wo_shoes")).alias("reach_to_height"),
])

print(f"Combined dataset: {len(combined):,} players with combine + career data")

# Correlation matrix
metric_cols = [
    "height_wo_shoes", "weight", "wingspan", "standing_reach",
    "body_fat_pct", "hand_length", "hand_width",
    "standing_vertical_leap", "max_vertical_leap",
    "lane_agility_time", "three_quarter_sprint", "bench_press",
    "wingspan_to_height", "reach_to_height",
]
outcome_cols = ["career_ppg", "career_gp", "career_pts", "seasons_played"]

# Compute correlations
corr_data = []
for metric in metric_cols:
    row = []
    for outcome in outcome_cols:
        valid = combined.filter(
            pl.col(metric).is_not_null() & pl.col(outcome).is_not_null()
        )
        if len(valid) > 10:
            r = np.corrcoef(
                valid[metric].to_numpy().astype(float),
                valid[outcome].to_numpy().astype(float),
            )[0, 1]
        else:
            r = np.nan
        row.append(round(r, 3))
    corr_data.append(row)

corr_array = np.array(corr_data)

# Pretty labels
metric_labels = [
    "Height (no shoes)", "Weight", "Wingspan", "Standing Reach",
    "Body Fat %", "Hand Length", "Hand Width",
    "Standing Vert", "Max Vert",
    "Lane Agility", "3/4 Sprint", "Bench Press",
    "Wingspan/Height", "Reach/Height",
]
outcome_labels = ["Career PPG", "Career GP", "Career PTS", "Seasons Played"]

fig = go.Figure(data=go.Heatmap(
    z=corr_array,
    x=outcome_labels,
    y=metric_labels,
    colorscale="RdBu",
    zmid=0,
    zmin=-0.4,
    zmax=0.4,
    text=np.round(corr_array, 2),
    texttemplate="%{text}",
    textfont=dict(size=11),
    hovertemplate="%{y} vs %{x}<br>r = %{z:.3f}<extra></extra>",
    colorbar=dict(title="Pearson r"),
))

fig.update_layout(
    title="Combine Metrics vs Career Outcomes: Pearson Correlations",
    height=600,
    width=700,
    yaxis=dict(autorange="reversed"),
)
fig.show()

takeaway(
    "The wingspan-to-height ratio correlates more strongly with career PPG and total "
    "points than raw wingspan alone. This suggests it is the *disproportionate* length "
    "relative to body size -- not just being tall -- that predicts scoring ability. "
    "Lane agility time (lower = better) and 3/4 court sprint show negative correlations "
    "with career success: faster, more agile athletes tend to have longer, more "
    "productive careers."
)

---
## 5. Survival Analysis: Career Longevity

How long do NBA careers last, and which combine metrics predict staying power?
We use Kaplan-Meier curves and Cox Proportional Hazards regression from the
[lifelines](https://lifelines.readthedocs.io/) library.

In [ ]:
# Build survival dataset
survival_raw = conn.sql("""
    SELECT
        d.person_id,
        d.player_name,
        d.round_number,
        d.overall_pick,
        d.wingspan,
        d.height_wo_shoes,
        d.weight,
        d.body_fat_pct,
        d.standing_vertical_leap,
        d.max_vertical_leap,
        d.lane_agility_time,
        d.three_quarter_sprint,
        d.bench_press,
        c.seasons_played,
        p.is_active
    FROM fact_draft d
    INNER JOIN agg_player_career c ON d.person_id = c.player_id
    INNER JOIN dim_player p ON d.person_id = p.player_id
    WHERE d.wingspan IS NOT NULL
""").pl()

# Event: career ended (not active). Censored if still active.
survival_raw = survival_raw.with_columns([
    # event = 1 means career ended (observed), 0 means still active (censored)
    pl.when(pl.col("is_active").eq(True)).then(0).otherwise(1).alias("event"),
    # Round group for KM curves
    pl.when(pl.col("round_number") == 1).then(pl.lit("Round 1"))
     .when(pl.col("round_number") == 2).then(pl.lit("Round 2"))
     .otherwise(pl.lit("Undrafted / Other")).alias("round_group"),
])

print(f"Survival dataset: {len(survival_raw):,} players")
print(f"  Still active (censored): {survival_raw.filter(pl.col('event') == 0).height}")
print(f"  Career ended (observed): {survival_raw.filter(pl.col('event') == 1).height}")

In [ ]:
# Kaplan-Meier curves by draft round
fig_km, ax_km = plt.subplots(figsize=(10, 6))

km_colors = {"Round 1": COLORS["primary"], "Round 2": COLORS["secondary"],
             "Undrafted / Other": COLORS["muted"]}

for group in ["Round 1", "Round 2", "Undrafted / Other"]:
    subset = survival_raw.filter(pl.col("round_group") == group)
    if len(subset) < 5:
        continue
    kmf = KaplanMeierFitter()
    kmf.fit(
        durations=subset["seasons_played"].to_numpy(),
        event_observed=subset["event"].to_numpy(),
        label=f"{group} (n={len(subset)})",
    )
    kmf.plot_survival_function(ax=ax_km, color=km_colors[group], linewidth=2)

ax_km.set_xlabel("Seasons Played", fontsize=13)
ax_km.set_ylabel("Survival Probability", fontsize=13)
ax_km.set_title("Career Survival by Draft Round (Kaplan-Meier)", fontsize=15)
ax_km.legend(fontsize=11)
ax_km.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Cox Proportional Hazards with combine metrics
cox_cols = [
    "seasons_played", "event",
    "height_wo_shoes", "weight", "wingspan",
    "body_fat_pct", "standing_vertical_leap", "max_vertical_leap",
    "lane_agility_time", "three_quarter_sprint", "bench_press",
]

# --- Transparency: how much data is lost to complete-case analysis ---
n_before = len(survival_raw)
cox_df = survival_raw.select(cox_cols).drop_nulls().to_pandas()
n_after = len(cox_df)
print(f"Cox PH dataset: {n_after} complete cases out of {n_before} total ({n_before - n_after} dropped, {(n_before - n_after)/n_before*100:.1f}% excluded)")
print(f"Columns with most missingness that drove exclusions:")
for col in cox_cols:
    if col not in ["seasons_played", "event"]:
        miss = survival_raw[col].null_count()
        if miss > 0:
            print(f"  {col}: {miss} missing ({miss/n_before*100:.1f}%)")

# --- Multiple Imputation Cox PH (sensitivity analysis) ---
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

print("\n--- Multiple Imputation Cox PH (sensitivity analysis) ---")
imputer = IterativeImputer(max_iter=10, random_state=42)
cox_cols_numeric = [c for c in cox_cols if c not in ["seasons_played", "event"]]
cox_raw_pd = survival_raw.select(cox_cols).to_pandas()

# Impute the numeric covariate columns only (not duration/event)
cox_imputed = cox_raw_pd.copy()
cox_imputed[cox_cols_numeric] = imputer.fit_transform(cox_raw_pd[cox_cols_numeric])

cph_imputed = CoxPHFitter()
cph_imputed.fit(cox_imputed, duration_col="seasons_played", event_col="event")
print(f"Imputed Cox PH dataset: {len(cox_imputed)} players (all {n_before} included)")
cph_imputed.print_summary()

# --- Primary: Complete-case Cox PH ---
print("\n--- Complete-Case Cox PH (primary analysis) ---")
print(f"Cox PH dataset: {n_after} players (complete cases)")

cph = CoxPHFitter()
cph.fit(
    cox_df,
    duration_col="seasons_played",
    event_col="event",
)

print("\nCox Proportional Hazards — Summary:")
cph.print_summary()

# --- Proportional Hazards Assumption Test (NB-021) ---
print("\n--- Proportional Hazards Assumption Test ---")
try:
    cph.check_assumptions(cox_df, p_value_threshold=0.05, show_plots=True)
except Exception as e:
    print(f"PH assumption test failed: {e}")
    print("Consider stratifying on variables that violate the PH assumption.")

# Plot hazard ratios
fig_hr, ax_hr = plt.subplots(figsize=(8, 5))
cph.plot(ax=ax_hr)
ax_hr.set_title("Cox PH Hazard Ratios: Combine Metrics", fontsize=14)
ax_hr.axvline(x=0, color="gray", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

takeaway(
    "The complete-case analysis retains only players with all combine measurements, "
    "which may bias toward a specific prospect type. The imputed analysis includes all "
    f"{n_before} players. Compare hazard ratios between the two models — substantial "
    "differences indicate selection bias in the complete-case results."
)

takeaway(
    "Lane agility time is the strongest combine predictor of career longevity in the "
    "Cox PH model. A higher (slower) lane agility time is associated with increased "
    "hazard of career ending sooner. This makes intuitive sense: lateral quickness is "
    "critical for NBA defense at every position, and players who cannot defend at the "
    "NBA level are the first to lose their roster spots."
)

---
## 6. Star Prediction: CatBoost Classifier

Can we predict whether a prospect will become a bust, rotation player, starter, or star
based purely on their combine measurements and draft position?

We define four outcome classes:
- **Bust**: < 3 seasons played
- **Rotation**: 3-6 seasons AND < 10 career PPG
- **Starter**: 7+ seasons OR 10+ career PPG
- **Star**: At least one All-Star selection

We use CatBoost as our primary classifier (works well with mixed feature types and
handles missing values natively). TabPFN v2 is an alternative for smaller datasets
but may not be available on Kaggle.

In [ ]:
# Get All-Star selections
all_stars = conn.sql("""
    SELECT DISTINCT player_id
    FROM fact_player_awards
    WHERE description ILIKE '%all-star%'
       OR description ILIKE '%all star%'
""").pl()

all_star_ids = set(all_stars["player_id"].to_list())
print(f"All-Star players found: {len(all_star_ids)}")

# Build classification dataset
clf_data = conn.sql("""
    SELECT
        d.person_id,
        d.player_name,
        d.overall_pick,
        d.height_wo_shoes,
        d.weight,
        d.wingspan,
        d.standing_reach,
        d.body_fat_pct,
        d.hand_length,
        d.hand_width,
        d.standing_vertical_leap,
        d.max_vertical_leap,
        d.lane_agility_time,
        d.three_quarter_sprint,
        d.bench_press,
        c.career_ppg,
        c.career_gp,
        c.seasons_played,
        p.is_active
    FROM fact_draft d
    INNER JOIN agg_player_career c ON d.person_id = c.player_id
    INNER JOIN dim_player p ON d.person_id = p.player_id
    WHERE d.wingspan IS NOT NULL
      AND p.is_active = false
""").pl()

# Derive ratios
clf_data = clf_data.with_columns([
    (pl.col("wingspan") / pl.col("height_wo_shoes")).alias("wingspan_to_height"),
    (pl.col("standing_reach") / pl.col("height_wo_shoes")).alias("reach_to_height"),
])

# Assign outcome classes
def classify_outcome(row):
    pid = row["person_id"]
    seasons = row["seasons_played"]
    ppg = row["career_ppg"] if row["career_ppg"] is not None else 0
    if pid in all_star_ids:
        return "star"
    elif seasons >= 7 or ppg >= 10:
        return "starter"
    elif seasons >= 3 and ppg < 10:
        return "rotation"
    else:
        return "bust"

outcomes = [classify_outcome(row) for row in clf_data.iter_rows(named=True)]
clf_data = clf_data.with_columns(pl.Series("outcome", outcomes))

print(f"\nClassification dataset: {len(clf_data)} retired players with combine data")
print("\nOutcome distribution:")
for cls in ["bust", "rotation", "starter", "star"]:
    n = clf_data.filter(pl.col("outcome") == cls).height
    print(f"  {cls:10s}: {n:4d} ({n/len(clf_data)*100:.1f}%)")

In [ ]:
# Feature matrix
feature_cols = [
    "overall_pick", "height_wo_shoes", "weight", "wingspan",
    "standing_reach", "body_fat_pct", "hand_length", "hand_width",
    "standing_vertical_leap", "max_vertical_leap",
    "lane_agility_time", "three_quarter_sprint", "bench_press",
    "wingspan_to_height", "reach_to_height",
]

X = clf_data.select(feature_cols).to_pandas()
y = clf_data["outcome"].to_pandas()

# Encode labels
label_map = {"bust": 0, "rotation": 1, "starter": 2, "star": 3}
label_names = ["bust", "rotation", "starter", "star"]
y_encoded = y.map(label_map)

# 5-fold stratified cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_accs = []
all_y_true = []
all_y_pred = []
best_model = None
best_acc = 0

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_encoded), 1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y_encoded.iloc[train_idx], y_encoded.iloc[val_idx]

    model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function="MultiClass",
        eval_metric="Accuracy",
        random_seed=42,
        verbose=0,
        nan_mode="Min",  # CatBoost handles NaN natively
    )
    model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50)

    y_pred = model.predict(X_val).flatten()
    acc = accuracy_score(y_val, y_pred)
    fold_accs.append(acc)
    all_y_true.extend(y_val.tolist())
    all_y_pred.extend(y_pred.tolist())

    if acc > best_acc:
        best_acc = acc
        best_model = model

    print(f"  Fold {fold}: Accuracy = {acc:.3f}")

print(f"\nMean CV Accuracy: {np.mean(fold_accs):.3f} +/- {np.std(fold_accs):.3f}")
print(f"\nClassification Report (all folds combined):")
print(classification_report(
    all_y_true, all_y_pred, target_names=label_names, digits=3
))

In [ ]:
# Confusion matrix heatmap
cm = confusion_matrix(all_y_true, all_y_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig = go.Figure(data=go.Heatmap(
    z=cm_pct,
    x=label_names,
    y=label_names,
    colorscale=[[0, "white"], [1, COLORS["primary"]]],
    text=[[f"{cm[i][j]}<br>({cm_pct[i][j]:.0f}%)" for j in range(4)] for i in range(4)],
    texttemplate="%{text}",
    textfont=dict(size=13),
    hovertemplate="True: %{y}<br>Predicted: %{x}<br>Count: %{z:.0f}%<extra></extra>",
    colorbar=dict(title="% of True"),
))
fig.update_layout(
    title="Confusion Matrix: CatBoost Star Prediction",
    xaxis_title="Predicted",
    yaxis_title="Actual",
    height=500,
    width=550,
    yaxis=dict(autorange="reversed"),
)
fig.show()

In [ ]:
# SHAP analysis — retrain on full data for SHAP values
full_model = CatBoostClassifier(
    iterations=500, learning_rate=0.05, depth=6,
    loss_function="MultiClass", random_seed=42, verbose=0, nan_mode="Min",
)
full_model.fit(X, y_encoded)

explainer = shap.TreeExplainer(full_model)
shap_values = explainer.shap_values(X)

# Global feature importance (mean |SHAP| across all classes)
mean_abs_shap = np.mean([np.abs(shap_values[c]).mean(axis=0) for c in range(4)], axis=0)
feat_importance = sorted(zip(feature_cols, mean_abs_shap), key=lambda x: -x[1])

print("Global Feature Importance (mean |SHAP|):")
for feat, imp in feat_importance:
    print(f"  {feat:25s}: {imp:.4f}")

In [ ]:
# SHAP waterfall plots for notable draft picks
# Find specific players for case studies
notable_cases = []
for outcome_class, label in [("star", "Star"), ("bust", "Bust")]:
    subset = clf_data.filter(pl.col("outcome") == outcome_class)
    if len(subset) >= 2:
        # Pick first two by name for reproducibility
        names = subset.sort("player_name").head(2)
        for row in names.iter_rows(named=True):
            notable_cases.append((row["player_name"], label))

fig_shap, axes_shap = plt.subplots(
    len(notable_cases), 1,
    figsize=(10, 4 * len(notable_cases)),
)
if len(notable_cases) == 1:
    axes_shap = [axes_shap]

for ax_i, (pname, plabel) in enumerate(notable_cases):
    idx_match = clf_data.with_row_index("idx").filter(
        pl.col("player_name") == pname
    )
    if len(idx_match) == 0:
        continue
    row_idx = idx_match["idx"][0]
    pred_class = int(full_model.predict(X.iloc[[row_idx]])[0])

    # Use SHAP values for the predicted class
    sv = shap_values[pred_class][row_idx]
    base = explainer.expected_value[pred_class]

    # Sort features by |SHAP|
    sorted_idx = np.argsort(np.abs(sv))[::-1][:10]
    ax = axes_shap[ax_i]
    feats = [feature_cols[i] for i in sorted_idx]
    vals = [sv[i] for i in sorted_idx]
    colors = [COLORS["secondary"] if v < 0 else COLORS["primary"] for v in vals]

    ax.barh(feats[::-1], [vals[i] for i in range(len(vals))][::-1], color=colors[::-1])
    ax.set_title(f"{pname} ({plabel}) - Predicted: {label_names[pred_class]}", fontsize=13)
    ax.set_xlabel("SHAP value")
    ax.axvline(x=0, color="gray", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

takeaway(
    "Draft position (overall_pick) is by far the most important predictor -- which "
    "makes sense since teams have far more information than just combine metrics. "
    "Among pure physical measurements, lane agility time, max vertical leap, and "
    "wingspan-to-height ratio are the strongest signals for separating stars from busts. "
    "CatBoost achieves reasonable accuracy but struggles most with the star class, "
    "confirming that stardom depends heavily on factors beyond physical tools."
)

---
## 7. Biggest Gaps: Outperformers & Underperformers

Who most defied their combine profile? We look at the residuals between predicted
outcome probability and actual career outcome to find the biggest surprises.

In [ ]:
# Get predicted probabilities from the full model
pred_probs = full_model.predict_proba(X)
pred_classes = full_model.predict(X).flatten().astype(int)

# Compute "gap" — difference between actual outcome rank and predicted
# outcome rank, weighted by confidence
actual_encoded = y_encoded.values

# Expected outcome = weighted sum of class indices by probability
expected_outcome = np.sum(pred_probs * np.arange(4), axis=1)
gap = actual_encoded - expected_outcome  # positive = outperformed

gap_df = clf_data.with_columns([
    pl.Series("predicted_class", [label_names[c] for c in pred_classes]),
    pl.Series("p_bust", np.round(pred_probs[:, 0], 3)),
    pl.Series("p_rotation", np.round(pred_probs[:, 1], 3)),
    pl.Series("p_starter", np.round(pred_probs[:, 2], 3)),
    pl.Series("p_star", np.round(pred_probs[:, 3], 3)),
    pl.Series("expected_rank", np.round(expected_outcome, 2)),
    pl.Series("gap", np.round(gap, 2)),
])

# Top outperformers (predicted bust/rotation, became star/starter)
outperformers = (
    gap_df
    .sort("gap", descending=True)
    .head(15)
    .select(["player_name", "overall_pick", "outcome", "predicted_class",
             "p_bust", "p_star", "career_ppg", "seasons_played", "gap"])
)

# Top underperformers (predicted star/starter, became bust/rotation)
underperformers = (
    gap_df
    .sort("gap")
    .head(15)
    .select(["player_name", "overall_pick", "outcome", "predicted_class",
             "p_bust", "p_star", "career_ppg", "seasons_played", "gap"])
)

print("BIGGEST OUTPERFORMERS (predicted low, achieved high):")
print(outperformers)
print()
print("BIGGEST UNDERPERFORMERS (predicted high, achieved low):")
print(underperformers)

In [ ]:
# Plotly tables for outperformers and underperformers
def make_gap_table(df, title, color):
    fig = go.Figure(data=[go.Table(
        header=dict(
            values=["Player", "Pick", "Actual", "Predicted", "P(Bust)",
                    "P(Star)", "PPG", "Seasons", "Gap"],
            fill_color=color,
            font=dict(color="white", size=12),
            align="center",
        ),
        cells=dict(
            values=[
                df["player_name"].to_list(),
                df["overall_pick"].to_list(),
                df["outcome"].to_list(),
                df["predicted_class"].to_list(),
                df["p_bust"].to_list(),
                df["p_star"].to_list(),
                [f"{v:.1f}" if v is not None else "" for v in df["career_ppg"].to_list()],
                df["seasons_played"].to_list(),
                df["gap"].to_list(),
            ],
            fill_color="white",
            align="center",
            font=dict(size=11),
        ),
    )])
    fig.update_layout(title=title, height=450, width=900)
    return fig

fig_out = make_gap_table(outperformers, "Biggest Outperformers: Defied Their Combine Profile", COLORS["success"])
fig_out.show()

fig_under = make_gap_table(underperformers, "Biggest Underperformers: Fell Short of Combine Promise", COLORS["secondary"])
fig_under.show()

takeaway(
    "The model's biggest misses highlight what combine data fundamentally cannot capture: "
    "basketball IQ, work ethic, injury resilience, team fit, and mental toughness. "
    "Outperformers often had elite intangibles that compensated for middling athleticism, "
    "while underperformers frequently had careers cut short by injury or struggled to "
    "translate raw tools into basketball production. The combine measures the ceiling, "
    "not the probability of reaching it."
)

---
## 8. Conclusion & Cross-Links

### Summary of Findings

1. **The combine athlete has evolved**: players are heavier but just as explosive as previous decades.
2. **Draft pick value drops steeply after ~pick 14**: lottery picks are in a different tier.
3. **Wingspan-to-height ratio beats raw wingspan** as a predictor of career scoring.
4. **Lane agility is the strongest combine predictor of career longevity** (survival analysis).
5. **CatBoost can separate busts from starters** with reasonable accuracy, but stardom prediction remains noisy — confirming that physical tools are necessary but not sufficient.
6. **The biggest outperformers and underperformers** remind us that intangibles, injury luck, and development trajectories matter enormously.

### Methodology Notes

- **Survival Analysis**: Kaplan-Meier + Cox PH from [lifelines](https://lifelines.readthedocs.io/). Active players are right-censored.
- **Classification**: [CatBoost](https://catboost.ai/) with native NaN handling. 5-fold stratified CV.
- **Explainability**: [SHAP](https://shap.readthedocs.io/) TreeExplainer for feature attribution.
- **Alternative**: [TabPFN v2](https://github.com/PriorLabs/TabPFN) is an excellent zero-shot tabular classifier if available in your environment.

### nbadb Notebook Suite

| Part | Notebook | Description |
|---|---|---|
| Part 1 | [MVP Prediction](./nba_mvp_predictor.ipynb) | MVP Prediction with Tracking & Synergy Data |
| Part 2 | [Data-Driven Player Archetypes](./nba_player_archetypes.ipynb) | Data-Driven Player Archetypes (UMAP + GMM) |
| Part 3 | [Game Outcome Prediction](./nba_game_prediction.ipynb) | Game Outcome Prediction (Stacking Ensemble) |
| **Part 4** | **Draft Combine Analysis** (this notebook) | **Draft Combine to Career Prediction** |
| Part 5 | [Defense Decoded](./nba_defense_decoded.ipynb) | Defense Decoded (Tracking + Hustle + Synergy PCA) |
| Part 6 | [Interactive Player Explorer](./nba_player_dashboard.ipynb) | Interactive Player Explorer |
| Part 7 | [Spatial Shot Analysis](./nba_shot_chart_analysis.ipynb) | Spatial Shot Analysis & 3-Point Revolution |
| Part 8 | [Player Similarity Engine](./nba_player_similarity.ipynb) | Player Similarity Engine (Cosine + Manhattan) |
| Part 9 | [Career Trajectory](./nba_aging_curves.ipynb) | Career Trajectory & Aging Curve Modeling |
| Part 10 | [Play-by-Play Insights](./nba_play_by_play_insights.ipynb) | Play-by-Play: Win Probability, Runs & Clutch |

---

**Dataset**: [wyattowalsh/basketball](https://www.kaggle.com/datasets/wyattowalsh/basketball) on Kaggle
**Documentation**: [nbadb.dev](https://nbadb.dev)
**Source**: [github.com/wyattowalsh/nbadb](https://github.com/wyattowalsh/nbadb)

In [ ]:
# Cleanup handled by nbadb_utils.get_connection() via atexit
print("Analysis complete.")